In [ ]:
device = "0"
model_name = "meta-llama/Llama-3.2-3B"
dataset_size = 200
max_length = 256

In [3]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../src')

In [4]:
import torch
from utils import load_model_and_tokenizer, load_c4

experiment = "ppl"

device = torch.device(f"cuda:{device}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model, tokenizer = load_model_and_tokenizer(model_name, device)

data = load_c4(tokenizer, dataset_size)
data[0]

KeyboardInterrupt: 

In [ ]:
import wandb
from tqdm import tqdm

from utils import (
    load_default_wepa_watermarker,
    load_default_exp_watermarker,
    load_default_kgw_watermarker,
    load_default_unbiased_watermarker,
    get_wat_name,
    load_unwatermarked_watermarker,
    init_wandb,
)

seed = 42

wats = [
    load_default_wepa_watermarker(
        model.config.vocab_size,
        device,
        degree=1,
        seed=seed,
    ),
    load_default_wepa_watermarker(
        model.config.vocab_size,
        device,
        degree=2,
        seed=seed,
    ),
    load_default_exp_watermarker(
        model.config.vocab_size,
        device,
        seed=seed,
    ),
    load_default_kgw_watermarker(
        model.config.vocab_size,
        device,
        seed=seed,
    ),
    load_default_unbiased_watermarker(
        model.config.vocab_size,
        device,
        seed=seed,
    ),
    load_unwatermarked_watermarker(),
]


def compute_ppl(model, input_ids, continuation_ids, device):
    # concat: [prompt | continuation]
    full_ids = torch.cat([input_ids, continuation_ids.unsqueeze(0)], dim=1).to(device)
    labels = full_ids.clone()
    # mask out the prompt so it's not counted in loss
    labels[:, : input_ids.size(1)] = -100  # -100 is ignored in HF loss

    with torch.no_grad():
        outputs = model(full_ids, labels=labels)
        neg_log_likelihood = outputs.loss.item()
    return torch.exp(torch.tensor(neg_log_likelihood)).item()


def evaluate_statistics(wat, max_length):
    print(f"max_length: {max_length}")

    ppls = []
    for i in tqdm(range(dataset_size)):
        for j in range(10):
            inputs = data[i].to(device)
            outputs = wat.generate(
                model,
                inputs,
                max_new_tokens=max_length,
            )
            outputs = outputs[0, inputs["input_ids"].size(1) :]

            if len(outputs) < max_length:
                output_text = tokenizer.decode(outputs, skip_special_tokens=True)
                print(
                    "Text length is less than max_length: ",
                    len(outputs),
                    # output_text,
                )
                continue

            prompt_ids = inputs["input_ids"]  # shape [1, prompt_len]
            continuation_ids = outputs  # your generated ids, shape [cont_len]
            ppl = compute_ppl(model, prompt_ids, continuation_ids, device)
            ppls.append(ppl)
            break

    return ppls


for wat in wats:
    print(f"Running experiment: {wat}")

    wat_name = get_wat_name(wat)
    run_name = f"{model.name_or_path}_{wat_name}"

    ppls = evaluate_statistics(wat, max_length)
    ppl_median = torch.median(torch.tensor(ppls)).item()
    ppl_mean = torch.mean(torch.tensor(ppls)).item()

    metrics = {}
    tables = {}
    tables["ppls"] = wandb.Table(data=[[p] for p in ppls], columns=["ppl"])

    print(f"median={ppl_median:.4f} ({len(ppls)} samples)")
    print(f"mean={ppl_mean:.4f} ({len(ppls)} samples)")

    run = init_wandb(
        project=f"WEPA-{experiment}",
        name=run_name,
        config={
            "experiment": experiment,
            "dataset_size": dataset_size,
            "model": model.name_or_path,
            "algo": wat_name,
            "text_length": max_length,
            "lam": getattr(wat, "lam", None),
            "bits": getattr(wat, "bits", None),
            "device": str(device),
        },
    )

    # Log scalar stats
    wandb.log(metrics)

    # Log tabular data
    for key, value in tables.items():
        wandb.log({key: value})

    run.finish()

Running experiment: <watermarker.wepa.WepaWatermarker object at 0x7f74fec45850>
max_length: 50


 24%|██▍       | 24/100 [00:17<00:55,  1.38it/s]

Text length is less than max_length:  10


 77%|███████▋  | 77/100 [00:56<00:16,  1.38it/s]

Text length is less than max_length:  44


...

Text length is less than max_length:  21


100%|██████████| 100/100 [01:15<00:00,  1.32it/s]


median=11.9505 (99 samples)
mean=15.0844 (99 samples)
Running experiment: <utils.UnwatermarkedWatermarker object at 0x7f702cd77a70>
max_length: 50


100%|██████████| 100/100 [01:11<00:00,  1.39it/s]

median=11.8673 (100 samples)
mean=15.9430 (100 samples)
